# **Lab Handouts**
## Module 2: Numerical Computation, Strings, Collections and a Mini Physics Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esha06/module-2-numerical-computation/blob/main/Module_2_Lab.ipynb)

**Topics covered:**

1. Numerical computation
2. Engineering formula evaluation
3. Demonstrating floating-point precision issues
4. String manipulation and formatting
5. Collection operations
6. Creating and accessing nested lists and dictionaries
7. Standard Library: the `math` module overview
8. Python String Methods reference (docs.python.org)
9. Physics Engine Fundamentals: implementing basic kinematic equations as Python expressions
10. Automated Report Generation: formatting raw scientific data into human-readable summaries using f-strings

**What you will produce:** this notebook, a `physics.py` module, a `report.py` module, a test
file, a generated `report.txt`, and a GitHub release called `v0.1.0`.

**How to work:** run each code cell with **Shift+Enter**, read the output, then do the
**Exercise** at the end of each topic in a new cell. Only the Python standard library is used —
nothing to install.

---
## 1. Numerical computation

Python has three built-in number types: `int` (whole numbers, unlimited size), `float`
(decimals, about 15–16 significant digits) and `complex`.

| Operator | Meaning | Example |
|---|---|---|
| `+ - *` | add, subtract, multiply | `7 * 3` → `21` |
| `/` | true division (always a float) | `7 / 2` → `3.5` |
| `//` | floor division | `7 // 2` → `3` |
| `%` | remainder (modulo) | `7 % 2` → `1` |
| `**` | power | `2 ** 10` → `1024` |

In [ ]:
a, b = 17, 5
print("a + b  =", a + b)
print("a / b  =", a / b, "  (true division gives a float)")
print("a // b =", a // b, "  (floor division)")
print("a % b  =", a % b, "  (remainder)")
print("a ** b =", a ** b)
print("divmod(a, b) =", divmod(a, b), "  (quotient and remainder together)")

In [ ]:
print("-7 // 2 =", -7 // 2, "  (floors toward minus infinity, not toward zero)")
print("-2 ** 2 =", -2 ** 2, "  (** binds tighter than unary minus)")
print("(-2) ** 2 =", (-2) ** 2)
print("2 ** 100 =", 2 ** 100, "  (ints never overflow)")
print("1e308 * 10 =", 1e308 * 10, "  (floats do)")
print("types:", type(3), type(3.0), type(3 + 4j))
print("int(7.9) =", int(7.9), "| round(7.5) =", round(7.5), "| round(8.5) =", round(8.5), "  (round half to even)")
print("1_000_000 + 1 =", 1_000_000 + 1, "  (underscores make big numbers readable)")

**Exercise 1:** A rope 1 000 cm long is cut into pieces of 37 cm. Using `//` and `%`, print how many
full pieces you get and how much rope is left over.

---
## 2. Engineering formula evaluation

An engineering formula becomes a Python expression almost word for word. Good habits:

* one variable per quantity, with a **descriptive name**;
* write the **unit in a comment** next to each input;
* keep everything in SI units, then convert only when you print.

In [ ]:
# Ohm's law (V = I*R) and electrical power (P = V*I)
voltage = 230.0       # V
resistance = 52.9     # ohm
current = voltage / resistance     # A
power = voltage * current          # W
print(f"Current = {current:.3f} A, Power = {power:.1f} W")

In [ ]:
# Reynolds number: Re = rho * v * D / mu  (tells laminar from turbulent pipe flow)
density = 998.0        # kg/m^3, water at 20 C
velocity = 1.5         # m/s
diameter = 0.05        # m
viscosity = 1.002e-3   # Pa*s

reynolds = density * velocity * diameter / viscosity
if reynolds < 2300:
    regime = "laminar"
elif reynolds > 4000:
    regime = "turbulent"
else:
    regime = "transitional"
print(f"Re = {reynolds:,.0f} -> {regime} flow")

In [ ]:
# Bending stress in a rectangular beam: sigma = M * y / I, with I = b*h^3/12
width = 0.10           # m
height = 0.20          # m
moment = 15_000        # N*m

second_moment = width * height ** 3 / 12   # m^4
y_max = height / 2                         # m, distance to the outer fibre
stress = moment * y_max / second_moment    # Pa
print(f"I = {second_moment:.3e} m^4, max stress = {stress / 1e6:.2f} MPa")

**Exercise 2:** The kinetic energy of a moving body is $E_k = \tfrac{1}{2} m v^2$. Compute it for a
1 200 kg car at 90 km/h (convert to m/s first!) and print the answer in kJ with one decimal place.

---
## 3. Demonstrating floating-point precision issues

Computers store floats in **binary**. Most decimal fractions (like 0.1) have no exact binary
form, so a tiny rounding error is stored with them. Usually harmless — but it breaks `==`
comparisons and can grow when you add many numbers.

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(f"0.1 is really stored as {0.1:.20f}")

In [ ]:
import math
import sys

# 1. Compare floats with a tolerance, never with ==
print("isclose(0.1 + 0.2, 0.3):", math.isclose(0.1 + 0.2, 0.3))

# 2. Errors accumulate when adding many small numbers
total = 0.0
for _ in range(10):
    total += 0.1
print("0.1 added ten times:", total, "| == 1.0?", total == 1.0, "| math.fsum:", math.fsum([0.1] * 10))

# 3. round() can surprise you
print("round(2.675, 2) =", round(2.675, 2), "  because 2.675 is stored as", f"{2.675:.20f}")

# 4. Adding a tiny number to a huge one loses the tiny one
print("(1e16 + 1) - 1e16 =", (1e16 + 1) - 1e16)

# 5. Machine epsilon: the gap between 1.0 and the next float
print("epsilon =", sys.float_info.epsilon)

In [ ]:
# When you need exact decimal or rational arithmetic (money, exact ratios):
from decimal import Decimal
from fractions import Fraction

print("Decimal:", Decimal("0.1") + Decimal("0.2"))
print("Fraction:", Fraction(1, 10) + Fraction(2, 10))
print("Decimal built from a float shows the stored error:", Decimal(0.1))

**Rule of thumb:** use `math.isclose()` to compare floats, `math.fsum()` for long sums, and
`Decimal` when exact decimal results matter (e.g. currency).

**Exercise 3:** Predict, then check: is `0.1 * 3 == 0.3`? Is `math.isclose(0.1 * 3, 0.3)`?

---
## 4. String manipulation and formatting

Strings are **immutable** sequences of characters: every method returns a *new* string.
A common task is cleaning up a messy line of raw data.

In [ ]:
raw = "   T-101 , reactor TEMPERATURE ,  351.2749 , kelvin  "

parts = [p.strip() for p in raw.split(",")]   # split on commas, trim spaces
print(parts)

tag, name, value, unit = parts                # unpack into variables
name = name.title()                           # "Reactor Temperature"
value = float(value)                          # text -> number
print(tag, "|", name, "|", value, "|", unit.upper())

In [ ]:
print(name.lower(), "/", name.upper(), "/", name.swapcase())
print(name.replace("Reactor", "Boiler"))
print("find('Temp') ->", name.find("Temp"), "| startswith('Reactor') ->", name.startswith("Reactor"))
print("slicing:", tag[:1], tag[-3:], name[::-1])
print("join:", " | ".join(parts))
print("len:", len(name), "| count('e'):", name.count("e"))

### f-strings: format specifiers

Inside `{}` you can add `:` followed by a **format spec**: width, alignment, precision and type.

In [ ]:
print(f"{name}: {value:.2f} K")                          # 2 decimal places
print(f"[{tag:<10}] [{tag:>10}] [{tag:^10}]")            # left, right, centre in width 10
print(f"{1234567.891:,.2f}")                              # thousands separator
print(f"{0.000123:.2e}")                                  # scientific notation
print(f"{0.4567:.1%}")                                    # percentage
print(f"{42:05d}  {255:#x}  {5:b}")                       # zero-padding, hex, binary
print(f"{value=}")                                        # self-documenting (Python 3.8+)
print("older styles: %.1f K" % value, "| {:.1f} K".format(value))

**Exercise 4:** Clean the line `"  p-07 ;   inlet pressure;101.325 ;KPA "` (note the `;`) and print it as
`P-07 Inlet Pressure = 101.33 kPa`.

---
## 5. Collection operations

| Type | Syntax | Ordered | Changeable | Duplicates |
|---|---|---|---|---|
| `list` | `[1, 2, 3]` | yes | yes | yes |
| `tuple` | `(1, 2, 3)` | yes | no | yes |
| `set` | `{1, 2, 3}` | no | yes | no |
| `dict` | `{"a": 1}` | yes (insertion) | yes | keys unique |

In [ ]:
readings = [23.1, 22.8, 24.5, 23.9]
readings.append(25.2)              # add one item
readings.extend([21.7, 22.0])      # add several
print(readings, "count =", len(readings))
print("first:", readings[0], "| last:", readings[-1], "| slice [1:4]:", readings[1:4])
print("sorted:", sorted(readings))
print("min/max/mean:", min(readings), max(readings), round(sum(readings) / len(readings), 2))
print("above 23 (list comprehension):", [r for r in readings if r > 23])

In [ ]:
point = (3.0, 4.0)          # tuple: a fixed record
x, y = point                # tuple unpacking
print("x =", x, "y =", y)

group_a = {"sensor1", "sensor2", "sensor3"}
group_b = {"sensor3", "sensor4"}
print("union:", group_a | group_b)
print("intersection:", group_a & group_b)
print("difference:", group_a - group_b)

In [ ]:
constants = {"g": 9.81, "c": 299_792_458}
constants["h"] = 6.626e-34                          # add a key
print("g =", constants["g"])
print("k =", constants.get("k", "not defined"))      # safe lookup with default
for key, val in constants.items():
    print(f"  {key} = {val}")

quantities = ["mass", "speed", "time"]
values = [2.0, 3.5, 10]
print(dict(zip(quantities, values)))                 # pair two lists into a dict
for i, q in enumerate(quantities, start=1):
    print(i, q)

**Exercise 5:** From `readings`, build a new list with each value converted from °C to °F
(`F = C * 9/5 + 32`) using a list comprehension, rounded to 1 decimal place.

---
## 6. Creating and accessing nested lists and dictionaries

Collections can contain other collections. Access them by chaining indexes/keys **left to right**.

In [ ]:
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]
print("row 1:", matrix[1])
print("row 1, column 2:", matrix[1][2])
print("first column:", [row[0] for row in matrix])
print("transpose:", [list(col) for col in zip(*matrix)])

In [ ]:
experiments = {
    "EXP-01": {"material": "steel", "temps_C": [20, 150, 300], "passed": True},
    "EXP-02": {"material": "aluminium", "temps_C": [20, 90], "passed": False},
}

print(experiments["EXP-01"]["material"])        # dict -> dict -> value
print(experiments["EXP-01"]["temps_C"][-1])     # dict -> dict -> list -> item

experiments["EXP-02"]["temps_C"].append(120)    # change a nested list
experiments["EXP-03"] = {"material": "copper", "temps_C": [20, 60], "passed": True}

for exp_id, info in experiments.items():
    print(f"{exp_id}: {info['material']:<10} max {max(info['temps_C']):>3} C  passed={info['passed']}")

In [ ]:
import json
print(json.dumps(experiments, indent=2))   # pretty-print any nested structure

In [ ]:
# A classic trap: [[0] * 3] * 3 repeats the SAME inner list three times
grid = [[0] * 3] * 3
grid[0][0] = 1
print("trap:   ", grid)

grid = [[0] * 3 for _ in range(3)]         # a new inner list each time
grid[0][0] = 1
print("correct:", grid)

**Exercise 6:** Add `"operator": "your name"` inside `EXP-01`, then print the average temperature of every
experiment that `passed`.

---
## 7. Standard Library: the `math` module overview

`import math` gives you constants and functions implemented in C (fast and accurate).
Full reference: [docs.python.org/3/library/math.html](https://docs.python.org/3/library/math.html)

| Group | Examples |
|---|---|
| Constants | `pi`, `e`, `tau`, `inf`, `nan` |
| Powers & roots | `sqrt`, `pow`, `exp`, `hypot`, `cbrt` (3.11+) |
| Logarithms | `log`, `log10`, `log2` |
| Trigonometry | `sin`, `cos`, `tan`, `atan2`, `radians`, `degrees` |
| Rounding | `floor`, `ceil`, `trunc` |
| Integers | `factorial`, `comb`, `perm`, `gcd` |
| Float checks | `isclose`, `isnan`, `isinf`, `fsum` |

In [ ]:
import math

print("pi, e, tau:", math.pi, math.e, math.tau)
print("sqrt(2):", math.sqrt(2), "| hypot(3, 4):", math.hypot(3, 4))
print("sin(30 deg):", math.sin(math.radians(30)), "| 45 deg in rad:", math.radians(45))
print("log10(1000):", math.log10(1000), "| ln(e):", math.log(math.e), "| log(8, 2):", math.log(8, 2))
print("floor/ceil/trunc of -2.5:", math.floor(-2.5), math.ceil(-2.5), math.trunc(-2.5))
print("5!:", math.factorial(5), "| C(5,2):", math.comb(5, 2), "| gcd(12,18):", math.gcd(12, 18))
print("isnan(nan):", math.isnan(math.nan), "| isinf(inf):", math.isinf(math.inf))

In [ ]:
# Everything the module offers:
print([name for name in dir(math) if not name.startswith("_")])

**Exercise 7:** Use `help(math.atan2)` to read its documentation, then compute the angle (in degrees) of the
vector (x = -1, y = 1).

---
## 8. Python String Methods reference (docs.python.org)

The official list of every `str` method is at
[docs.python.org/3/library/stdtypes.html#string-methods](https://docs.python.org/3/library/stdtypes.html#string-methods).
The ones you will use most:

| Method | What it does | Example → result |
|---|---|---|
| `strip()` | remove surrounding whitespace | `"  hi ".strip()` → `"hi"` |
| `lower()` / `upper()` | change case | `"Hi".upper()` → `"HI"` |
| `title()` | capitalise each word | `"flow rate".title()` → `"Flow Rate"` |
| `split(sep)` | text → list | `"a,b".split(",")` → `["a", "b"]` |
| `join(list)` | list → text | `"-".join(["a", "b"])` → `"a-b"` |
| `replace(old, new)` | substitute | `"1,5".replace(",", ".")` → `"1.5"` |
| `find(sub)` | index or -1 | `"abc".find("c")` → `2` |
| `startswith` / `endswith` | test prefix/suffix | `"data.csv".endswith(".csv")` → `True` |
| `isdigit()` | only digits? | `"42".isdigit()` → `True` |
| `zfill(n)` | pad with zeros | `"7".zfill(3)` → `"007"` |
| `center(n)` / `ljust` / `rjust` | pad to width | `"ok".center(6, "*")` → `"**ok**"` |
| `count(sub)` | occurrences | `"banana".count("a")` → `3` |

In [ ]:
string_methods = [m for m in dir(str) if not m.startswith("_")]
print(len(string_methods), "public str methods:")
print(string_methods)

In [ ]:
help(str.split)   # built-in documentation for any method

**Exercise 8:** Pick two methods from the list above that are **not** in the table, look them up on
docs.python.org, and write a one-line example of each.

---
## 9. Physics Engine Fundamentals: kinematic equations as Python expressions

For motion with **constant acceleration** $a$, starting velocity $u$, time $t$ and displacement $s$:

| Equation | Python |
|---|---|
| $v = u + at$ | `u + a * t` |
| $s = ut + \tfrac{1}{2}at^2$ | `u * t + 0.5 * a * t ** 2` |
| $v^2 = u^2 + 2as$ | `math.sqrt(u ** 2 + 2 * a * s)` |

We put them in a module, `physics.py`, so they can be reused and tested.
`%%writefile` on the first line of a cell saves the cell to a file instead of running it.

In [ ]:
%%writefile physics.py
"""Basic kinematics for constant acceleration (SI units: m, s, m/s, m/s^2)."""
import math

G = 9.81  # gravitational acceleration, m/s^2


def final_velocity(u, a, t):
    """v = u + a*t"""
    return u + a * t


def displacement(u, a, t):
    """s = u*t + 1/2*a*t^2"""
    return u * t + 0.5 * a * t ** 2


def velocity_from_displacement(u, a, s):
    """v = sqrt(u^2 + 2*a*s)"""
    v_squared = u ** 2 + 2 * a * s
    if v_squared < 0:
        raise ValueError("u**2 + 2*a*s is negative: the object never reaches s")
    return math.sqrt(v_squared)


def projectile(speed, angle_deg, g=G):
    """Launch from ground level; return time of flight, max height and range."""
    theta = math.radians(angle_deg)
    vx = speed * math.cos(theta)
    vy = speed * math.sin(theta)
    time_of_flight = 2 * vy / g
    return {
        "time_of_flight": time_of_flight,
        "max_height": vy ** 2 / (2 * g),
        "range": vx * time_of_flight,
    }


def simulate_fall(height, dt=0.01, g=G):
    """Step a dropped object forward in time until it reaches the ground.

    This is how a physics engine works: instead of a formula, it updates
    velocity and position in many small time steps. Returns (time, speed).
    """
    t, y, v = 0.0, height, 0.0
    while y > 0:
        v += g * dt
        y -= v * dt
        t += dt
    return t, v

In [ ]:
import importlib
import physics
importlib.reload(physics)   # picks up your edits if you re-run the %%writefile cell

t = 3.0   # s
print(f"Free fall for {t} s: v = {physics.final_velocity(0, physics.G, t):.2f} m/s, "
      f"distance = {physics.displacement(0, physics.G, t):.2f} m")
print(f"Speed after falling 20 m: {physics.velocity_from_displacement(0, physics.G, 20):.2f} m/s")

# A car braking from 25 m/s at -6 m/s^2: how long and how far until it stops?
u, a = 25.0, -6.0
stop_time = -u / a
print(f"Braking: stops after {stop_time:.2f} s and {physics.displacement(u, a, stop_time):.1f} m")

In [ ]:
print("Projectile launched at 20 m/s")
print(f"{'Angle':>5} | {'Flight (s)':>10} | {'Height (m)':>10} | {'Range (m)':>9}")
for angle in range(15, 90, 15):
    r = physics.projectile(20, angle)
    print(f"{angle:>5} | {r['time_of_flight']:>10.2f} | {r['max_height']:>10.2f} | {r['range']:>9.2f}")

A real **physics engine** (games, simulators) does not use the closed-form formulas. It advances the
world in tiny **time steps** `dt`: update velocity from acceleration, then position from velocity.
Smaller steps → closer to the exact answer, but more computation.

In [ ]:
import math

height = 20.0                                   # m
exact = math.sqrt(2 * height / physics.G)       # from s = 1/2 g t^2
for dt in (0.1, 0.01, 0.001):
    t_sim, v_sim = physics.simulate_fall(height, dt)
    print(f"dt = {dt:<6} -> t = {t_sim:.4f} s  (exact {exact:.4f} s, error {abs(t_sim - exact):.4f} s)")

**Exercise 9:** Which launch angle in the table gives the longest range? Add a `projectile` call for the
Moon (`g=1.62`) at that angle and compare.

---
## 10. Automated Report Generation with f-strings

Raw data is lists of numbers. A report turns it into a **summary a human can read**: aligned columns,
consistent decimal places, units, and a conclusion. `report.py` does this with f-strings only.

In [ ]:
%%writefile report.py
"""Turn raw scientific measurements into a readable text report."""
import math


def summarize(values):
    """Return count, mean, min, max and (population) standard deviation."""
    n = len(values)
    if n == 0:
        raise ValueError("no values to summarize")
    mean = sum(values) / n
    variance = sum((x - mean) ** 2 for x in values) / n
    return {
        "n": n,
        "mean": mean,
        "min": min(values),
        "max": max(values),
        "std": math.sqrt(variance),
    }


def format_report(title, datasets, decimals=2):
    """datasets maps a quantity name to {"unit": str, "values": [numbers]}."""
    header = f"{'Quantity':<14}{'Unit':<7}{'n':>3}{'Mean':>10}{'Min':>10}{'Max':>10}{'Std':>8}"
    rule = "-" * len(header)
    lines = [title, "=" * len(title), header, rule]
    for name, data in datasets.items():
        s = summarize(data["values"])
        lines.append(
            f"{name:<14}{data['unit']:<7}{s['n']:>3}"
            f"{s['mean']:>10.{decimals}f}{s['min']:>10.{decimals}f}"
            f"{s['max']:>10.{decimals}f}{s['std']:>8.{decimals}f}"
        )
    total = sum(len(d["values"]) for d in datasets.values())
    lines.append(rule)
    lines.append(f"{len(datasets)} quantities, {total} readings in total.")
    return "\n".join(lines)

In [ ]:
import report
importlib.reload(report)

raw_data = {
    "Temperature": {"unit": "C", "values": [21.43, 21.97, 22.61, 23.08, 22.40]},
    "Pressure": {"unit": "kPa", "values": [101.3, 101.1, 100.9, 101.4, 101.2]},
    "Flow rate": {"unit": "L/min", "values": [12.5, 12.9, 13.4, 12.2, 12.8]},
}

text = report.format_report("Cooling Loop Test - Run 7", raw_data)
print(text)

In [ ]:
# Add a sentence-style conclusion, then save everything to a file
temps = report.summarize(raw_data["Temperature"]["values"])
rise = raw_data["Temperature"]["values"][-1] - raw_data["Temperature"]["values"][0]
conclusion = (
    f"Temperature averaged {temps['mean']:.1f} C (range {temps['min']:.1f}-{temps['max']:.1f} C) "
    f"and rose by {rise:+.2f} C over the run."
)
print(conclusion)

with open("report.txt", "w") as f:
    f.write(text + "\n\n" + conclusion + "\n")
print("Saved report.txt")

**Exercise 10:** Call `report.format_report` with `decimals=3`, and add a fourth quantity of your own
(e.g. `"Voltage"` in `V`) to `raw_data`.

---
## Check your work: unit tests

`test_module2.py` checks the physics and report functions automatically. All tests must pass
before you submit.

In [ ]:
%%writefile test_module2.py
import math
import unittest

from physics import (G, displacement, final_velocity, projectile,
                     simulate_fall, velocity_from_displacement)
from report import format_report, summarize


class TestKinematics(unittest.TestCase):
    def test_final_velocity(self):
        self.assertAlmostEqual(final_velocity(0, 9.81, 2), 19.62)

    def test_displacement(self):
        self.assertAlmostEqual(displacement(5, 2, 3), 24.0)

    def test_velocity_from_displacement(self):
        self.assertAlmostEqual(velocity_from_displacement(0, 9.81, 20), math.sqrt(392.4))

    def test_unreachable_height_raises(self):
        with self.assertRaises(ValueError):
            velocity_from_displacement(1, -9.81, 10)

    def test_projectile_range_at_45_degrees(self):
        self.assertAlmostEqual(projectile(20, 45)["range"], 20 ** 2 / G)

    def test_simulation_matches_formula(self):
        t, _ = simulate_fall(20, dt=0.001)
        self.assertAlmostEqual(t, math.sqrt(2 * 20 / G), delta=0.01)


class TestReport(unittest.TestCase):
    def test_summarize(self):
        s = summarize([1.0, 2.0, 3.0])
        self.assertEqual(s["n"], 3)
        self.assertAlmostEqual(s["mean"], 2.0)
        self.assertAlmostEqual(s["std"], math.sqrt(2 / 3))

    def test_summarize_empty_raises(self):
        with self.assertRaises(ValueError):
            summarize([])

    def test_format_report(self):
        text = format_report("Run 1", {"Temperature": {"unit": "C", "values": [20.0, 22.5]}})
        self.assertIn("Run 1", text)
        self.assertIn("21.25", text)
        self.assertIn("2 readings", text)


if __name__ == "__main__":
    unittest.main()

In [ ]:
!python -m unittest -v test_module2

You should see `Ran 9 tests` and `OK`.

---
## Submit your work on GitHub

Same workflow as Module 1:

1. Create a repository `module-2-numerical-computation` (Add README, `.gitignore`: Python) and a branch
   `feature/numerical-computation`.
2. **File → Save a copy in GitHub** → that repository, branch `feature/numerical-computation`,
   path `Module_2_Lab.ipynb`.
3. Run the cell below to download `physics.py`, `report.py` and `test_module2.py`, then upload them to the
   same branch (**Add file → Upload files**).
4. Open a pull request into `main`, review it, merge it.
5. **Releases → Draft a new release** → tag `v0.1.0`, title `Module 2 Lab v0.1.0` → **Publish release**.

In [ ]:
from google.colab import files

for name in ("physics.py", "report.py", "test_module2.py"):
    files.download(name)

---
## Reading
* [Python tutorial: Floating-Point Arithmetic — Issues and Limitations](https://docs.python.org/3/tutorial/floatingpoint.html)
* [`math` — Mathematical functions](https://docs.python.org/3/library/math.html)
* [String methods](https://docs.python.org/3/library/stdtypes.html#string-methods)
* [Format Specification Mini-Language](https://docs.python.org/3/library/string.html#formatspec)
* [Data structures (lists, tuples, sets, dicts)](https://docs.python.org/3/tutorial/datastructures.html)